In [1]:
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("../data/raw/twcs.csv")
print("Total rows:", len(df))
df.head()

Total rows: 2811774


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB


In [4]:
print("=== Sample INBOUND=True (customer messages hone chahiye) ===")
print(df[df["inbound"] == True][["author_id", "text"]].head(5).to_string())

print("\n=== Sample INBOUND=False (brand/support messages hone chahiye) ===")
print(df[df["inbound"] == False][["author_id", "text"]].head(5).to_string())

=== Sample INBOUND=True (customer messages hone chahiye) ===
  author_id                                                                                       text
1    115712                                              @sprintcare and how do you propose we do that
2    115712         @sprintcare I have sent several private messages and no one is responding as usual
4    115712                                                                         @sprintcare I did.
6    115712                                                  @sprintcare is the worst customer service
8    115713  @sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯

=== Sample INBOUND=False (brand/support messages hone chahiye) ===
    author_id                                                                                                                                  text
0  sprintcare             @115712 I understand. I would like to assist you. We would need to get you into

In [5]:
inbound_authors = set(df[df["inbound"] == True]["author_id"].unique())
outbound_authors = set(df[df["inbound"] == False]["author_id"].unique())
overlap = inbound_authors & outbound_authors

print("Unique customer-side authors:", len(inbound_authors))
print("Unique brand-side authors:", len(outbound_authors))
print("Authors in BOTH sets:", len(overlap))
print(list(overlap)[:20])

Unique customer-side authors: 702669
Unique brand-side authors: 108
Authors in BOTH sets: 0
[]


In [6]:
outbound_only = outbound_authors - inbound_authors
candidate_brands = df[df["author_id"].isin(outbound_only)]["author_id"].value_counts()

print("Total pure brand-side accounts:", len(candidate_brands))
candidate_brands.head(30)

Total pure brand-side accounts: 108


author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
VerizonSupport      17966
UPSHelp             17817
ATVIAssist          17650
O2                  16212
Safaricom_Care      16077
idea_cares          15724
AskTarget           13218
AirAsiaSupport      12829
BofA_Help           12683
SW_Help             12231
Name: count, dtype: int64

In [7]:
tweet_to_author = df.set_index("tweet_id")["author_id"]

top20 = candidate_brands.head(20).index
brand_tweets = df[df["author_id"].isin(top20)].copy()
brand_tweets["replied_to_author"] = brand_tweets["in_response_to_tweet_id"].map(tweet_to_author)

summary = brand_tweets.groupby("author_id").agg(
    total_replies=("tweet_id", "count"),
    unique_customers_helped=("replied_to_author", "nunique")
).sort_values("total_replies", ascending=False)

summary

,total_replies,unique_customers_helped
author_id,,
AmazonHelp,169840,71049
AppleSupport,106860,76366
Uber_Support,56270,38300
SpotifyCares,43265,27794
Delta,42253,22331
Tesco,38573,15594
AmericanAir,36764,21686
TMobileHelp,34317,19943
comcastcares,33031,21824


In [8]:
shortlist = ["AmazonHelp", "AppleSupport", "Uber_Support"]
shortlist_tweets = df[df["author_id"].isin(shortlist)].copy()
shortlist_tweets["text_len"] = shortlist_tweets["text"].str.len()
shortlist_tweets.groupby("author_id")["text_len"].describe()[["mean","min","25%","50%","max"]]

,mean,min,25%,50%,max
author_id,,,,,
AmazonHelp,123.963895,7.0,98.0,123.0,305.0
AppleSupport,136.612690,16.0,113.0,129.0,310.0
Uber_Support,110.070997,9.0,90.0,104.0,288.0


In [9]:
for brand in shortlist:
    print(f"\n=== {brand} sample ===")
    print(df[df["author_id"] == brand][["text"]].sample(10, random_state=42).to_string())


=== AmazonHelp sample ===
                                                                                                                                                                                                         text
1580471                                                                                                                                                           @523365 Ok, please keep us posted here. ^MC
658246                                                                                                  @193739 Merci pour votre commentaire 😊 Vous avez pu résoudre le souci de la commande non livrée ? ^MH
1412806                                                                                                                                                 @469296 What was advised when you contacted them? ^PK
1373951                                                                                        @470927 I'm sorry for this wait. What was the delivery

In [10]:
# Step 1: Amazon ke saare replies ke tweet_ids
amazon_replies = df[df["author_id"] == "AmazonHelp"]

# Step 2: Jin customer tweets ko Amazon ne reply kiya, unke IDs
customer_msg_ids = amazon_replies["in_response_to_tweet_id"].dropna().unique()

print("Amazon replies:", len(amazon_replies))
print("Customer messages Amazon replied to:", len(customer_msg_ids))

Amazon replies: 169840
Customer messages Amazon replied to: 155445


In [11]:
# tweet_id -> row ka quick lookup banate hai (fast access ke liye)
tweet_lookup = df.set_index("tweet_id")

def find_root(tweet_id, lookup, max_hops=20):
    """Thread ko peeche traverse karke sabse pehla tweet (root) dhoondo"""
    current = tweet_id
    hops = 0
    while hops < max_hops:
        row = lookup.loc[current] if current in lookup.index else None
        if row is None:
            return current  # broken thread, yahi ruk jao
        parent = row["in_response_to_tweet_id"]
        if pd.isna(parent):
            return current  # ye root hai
        current = int(parent)
        hops += 1
    return current  # max hops cross ho gaya (safety)

# Sirf test ke liye pehle 100 par try karo (poore dataset pe chalane se pehle)
sample_roots = [find_root(tid, tweet_lookup) for tid in customer_msg_ids[:100]]
print(sample_roots[:10])

[272.0, 272, 272, 325.0, 617.0, 617, 621.0, 624.0, 624, 630]


In [12]:
import time
start = time.time()

roots = {}
for tid in customer_msg_ids:
    roots[tid] = find_root(tid, tweet_lookup)

print("Done in", time.time() - start, "seconds")
print("Unique conversation roots found:", len(set(roots.values())))

Done in 17.24535632133484 seconds
Unique conversation roots found: 83478


In [15]:
import json

flagged_reasons = {}
with open("../data/processed/amazon_conversations_flagged.jsonl", encoding="utf-8") as f:
    for line in f:
        conv = json.loads(line)
        reason = conv.get("flag_reason", "unknown")
        flagged_reasons[reason] = flagged_reasons.get(reason, 0) + 1

print("Flag reason breakdown:")
for reason, count in sorted(flagged_reasons.items(), key=lambda x: -x[1]):
    print(f"  {reason}: {count}")

Flag reason breakdown:
  non_english: 20619
  empty_turn: 900


In [17]:
count = 0
with open("../data/processed/amazon_conversations_flagged.jsonl", encoding="utf-8") as f:
    for line in f:
        conv = json.loads(line)
        if conv.get("flag_reason") == "non_english":
            first_customer = next(t for t in conv["turns"] if t["role"] == "CUSTOMER")
            print(repr(first_customer["text"]))
            count += 1
        if count >= 15:
            break

"Quelqu'un peut m'aider à configurer une manette pour PC acheté sur @120533 ?\n\nReconnu par mon PC mais aucune réaction des touches ☹️"
'@115850 ग्राहकों कृपया Amazon से ख़रीदारी ना करे.पिछले20 दिन से pick up नहीं हुआ.last 7 दिन सेमैं ख़ुदट्राईकर रहा हूँ .आज किसीतरह हुआ .1/3'
'Hola @116928. ¿Cómo hablo con un responsable o encargado de un pedido de Prime? Por telefono es la tercera vez que cuelgan...'
'Hallo @116316, kurze Frage: die von mir als „Prime“ erworbene Lieferung „Next business day“ gibt es jetzt gar nicht mehr?'
'おい、Amazonさん\nここ海外だぞ\n交換を言ったんじゃないよ\nこれから包装ちゃんとしてくださいと言ったんだから！ https://t.co/e5gxHyYpUz'
'@AmazonHelp Pourquoi un portefeuille est livrable en point relais mais pas dans un amazon locker ?? 🙄'
'@115850 22 दिन से सिर्फ़ सॉरी बोला जा रहा है लेकिन मेरा रीफ़ंड नहीं हुआ .आप फ़ोन भी ख़रीदो जमा भी करो और प्राइम मेम्बर्शिप के पैसे भी दो'
'@AmazonHelp @115850 https://t.co/UUe7A6dAFL'
'@118919 https://t.co/z5ltPxIa45'
'amazonのマーケットプレイスで頼んだものと別のがきた。。。'
'Amazonキャンセルされたから直接買ったほうが早

In [19]:
with open("../data/processed/amazon_conversations_clean.jsonl", encoding="utf-8") as f:
    first_line = f.readline()

import json
sample = json.loads(first_line)
for turn in sample["turns"]:
    print("ORIGINAL:", turn["text"])
    print("CLEANED :", turn["cleaned_text"])
    print("---")

ORIGINAL: @115830 I just inadvertently bought a Kindle book on my account when I wanted to buy it as a gift, anyway I can change this?
CLEANED : I just inadvertently bought a Kindle book on my account when I wanted to buy it as a gift, anyway I can change this?
---
ORIGINAL: @356780 Oh no! Please give us a call here: https://t.co/JzP7hlA23B so we can take a look at this with you! ^TR
CLEANED : Oh no! Please give us a call here: so we can take a look at this with you!
---
